# Strands Agents with Bedrock AgentCore Memory — FSI Edition

This lab demonstrates how persistent memory transforms AI agents from stateless tools into context-aware advisors that remember client details across sessions.

## What You'll Build

1. Create an AgentCore Memory with three extraction strategies
2. Store rich FSI client conversations
3. Query each strategy independently and understand what it extracts
4. Build a memory-enabled agent that recalls client context
5. Demonstrate session handover — new TAM gets full client briefing

## Memory Strategies Explained

| Strategy | What It Extracts | FSI Example |
|----------|-----------------|-------------|
| **Summary** | Compressed session overview | "Discussed Acme Super's EKS migration timeline and latency requirements" |
| **User Preference** | Behavioral patterns & preferences | "Client prefers ESG investments, moderate risk appetite" |
| **Semantic** | Factual statements from user messages | "Acme Super spends $1.2M/month on AWS", "CTO is John Chen" |

## Setup

In [1]:
import boto3

region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')


Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


## What is Bedrock AgentCore Memory?

### Short-Term Memory

Short-term memory captures raw interaction events, maintains immediate context, powers real-time conversations, enriches long-term memory systems, and enables building advanced contextual solutions such as multi-step task completion, in-session knowledge accumulation, and context-aware decision making.

- **Synchronous storage** - Messages saved immediately during conversations
- **Configurable retention** - Automatically expires after specified period (from 7 to 365 days)

### Long-Term Memory

Long-term memory stores structured information extracted from raw agent interactions, which is retained across multiple sessions. Instead of saving all raw conversation data, long-term memory preserves only the key insights such as summaries of the conversations, facts and knowledge (semantic memory), or user preferences.

The memory pipeline is an asynchronous process that runs in the background and automatically extracts insights after raw conversation/context is stored in Short Term Memory via CreateEvent. This efficiently consolidates key information without interrupting live interactions.


## The Problem: Agents Forget Everything

Without persistent memory, every session starts blank:

In [5]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are a financial advisor assistant. Be concise.',
)

# This will fail - agent has no memory of any client
agent('What is Acme Super\'s monthly AWS spend and who is their CTO?')


Sorry, I can't share private company data. If you're an employee, contact your IT or finance department.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Sorry, I can't share private company data. If you're an employee, contact your IT or finance department."}], 'metadata': {'usage': {'inputTokens': 25, 'outputTokens': 26, 'totalTokens': 51}, 'metrics': {'latencyMs': 513, 'timeToFirstByteMs': 320}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[0.6210870742797852], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='babeafda-5516-4471-aa66-301641657fcf', usage={'inputTokens': 25, 'outputTokens': 26, 'totalTokens': 51})], usage={'inputTokens': 25, 'outputTokens': 26, 'totalTokens': 51})], traces=[<strands.telemetry.metrics.Trace object at 0x1309877d0>], accumulated_usage={'inputTokens': 25, 'outputTokens': 26, 'totalTokens': 51}, accumulated_metrics={'latencyMs': 513}), state={}, interrupts=None, structured_output=None)

The agent gives a generic answer because it has **zero context**. Let's fix that.

---

## Step 1: Reset Memory (for clean demo)

Run this to delete any existing memory from previous runs:

In [4]:
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=region)
memories = memory_client.list_memories()

for m in memories:
    mid = m['id']
    if 'FSI' in mid:
        print(f'Deleting: {mid}')
        memory_client.delete_memory_and_wait(memory_id=mid)
        print(f'  ✅ Deleted')

print('Ready for fresh start')


Ready for fresh start


## Step 2: Create Memory

We create a memory with three strategies. Each extracts different information from conversations.

⏱️ **Takes ~3 minutes to provision.**

In [7]:
from bedrock_agentcore.memory.constants import StrategyType
from botocore.exceptions import ClientError

MEMORY_NAME = 'FSI_ClientMemory'
ACTOR_ID = 'tam_zohaib'

print('⏱️ Creating memory (takes ~3 minutes)...')
try:
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description='FSI client context memory. All extractions must be in English.',
        strategies=[
            {
                StrategyType.SUMMARY.value: {
                    'name': 'SessionSummary',
                    'description': 'Summarize sessions in English. Capture key decisions, action items, and client requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/summaries/{{sessionId}}']
                }
            },
            {
                StrategyType.USER_PREFERENCE.value: {
                    'name': 'ClientPreferences',
                    'description': 'Extract client preferences in English. Focus on investment policy, risk appetite, technology preferences, and operational requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/preferences']
                }
            },
            {
                StrategyType.SEMANTIC.value: {
                    'name': 'ClientFacts',
                    'description': 'Extract factual statements in English. Focus on spend figures, contacts, architecture details, dates, and requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/facts/']
                }
            },
        ],
        event_expiry_days=30,
    )
    memory_id = memory.get('id')
    print(f'✅ Memory created: {memory_id}')
except ClientError as e:
    if 'already exists' in str(e):
        memories = memory_client.list_memories()
        memory_id = next(m['id'] for m in memories if MEMORY_NAME in m['id'])
        print(f'✅ Using existing: {memory_id}')
    else:
        raise e


Failed to create memory: An error occurred (ValidationException) when calling the CreateMemory operation: Validation failed during CreateMemory: Memory with name FSI_ClientMemory already exists


⏱️ Creating memory (takes ~3 minutes)...
✅ Using existing: FSI_ClientMemory-tt8CNnAfeL


## Step 3: Store Rich Client Conversations

We store detailed conversations about two FSI clients. The memory pipeline will automatically extract summaries, preferences, and facts.

In [8]:
# === SESSION 1: Acme Super Onboarding ===
memory_client.create_event(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id='acme-super-001',
    messages=[
        ('I have been assigned Acme Super as my new client. They are a superannuation fund managing $220 billion in assets.', 'USER'),
        ('That is a significant account. What are their primary workloads on AWS?', 'ASSISTANT'),
        ('Their core trading platform runs on EC2 with Oracle on RDS. They want to migrate to EKS with Aurora PostgreSQL by Q3 2026. The CTO John Chen is sponsoring this migration. His email is john.chen@acmesuper.com.au.', 'USER'),
        ('Noted the EKS migration target for Q3 2026, sponsored by CTO John Chen.', 'ASSISTANT'),
        ('Acme Super requires sub-10ms latency for trade execution in ap-southeast-2. They need 99.99 percent availability. Their DR site is us-west-2 with 15-minute RPO.', 'USER'),
        ('Critical NFRs captured: sub-10ms latency, 99.99% availability, DR in us-west-2 with 15-min RPO.', 'ASSISTANT'),
        ('Their investment policy is ESG-only. They refuse any exposure to fossil fuels, gambling, or weapons. They report to APRA quarterly. Their risk appetite is moderate.', 'USER'),
        ('ESG-only policy noted with APRA quarterly reporting and moderate risk appetite.', 'ASSISTANT'),
        ('Acme Super currently spends $1.2 million per month on AWS. EC2 is 45 percent of spend, RDS is 25 percent. They have zero Reserved Instances which is a big optimization opportunity.', 'USER'),
        ('Significant RI/SP opportunity on $1.2M monthly spend.', 'ASSISTANT'),
        ('The trading platform handles 50000 transactions per second at peak. They use Kafka for streaming and Redis for caching. The backup contact is Sarah Liu, Head of Platform Engineering.', 'USER'),
        ('Architecture: 50K TPS, Kafka, Redis. Backup contact: Sarah Liu (Head of Platform Eng).', 'ASSISTANT'),
    ],
)
print('✅ Session 1: Acme Super onboarding stored')

# === SESSION 2: Z-Pay Onboarding ===
memory_client.create_event(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id='zpay-001',
    messages=[
        ('My other client Z-Pay is a BNPL fintech. They process 5 million transactions daily and need real-time fraud detection under 100ms. Their fraud engine runs on SageMaker with custom models.', 'USER'),
        ('Z-Pay: 5M daily transactions, sub-100ms fraud detection on SageMaker.', 'ASSISTANT'),
        ('Z-Pay is very worried about upcoming ASIC regulations on BNPL. Their legal team requires all transaction data stays in Australia. Nothing can leave ap-southeast-2.', 'USER'),
        ('Data sovereignty: ap-southeast-2 only. ASIC regulatory concern noted.', 'ASSISTANT'),
        ('They spend $800K per month on AWS. They prefer Graviton instances for cost savings. DR is in us-west-2 with 5-minute RPO. The main contact is David Park, VP Engineering, david.park@zpay.com.', 'USER'),
        ('Z-Pay: $800K/month, Graviton preference, DR us-west-2 (5-min RPO), contact David Park.', 'ASSISTANT'),
    ],
)
print('✅ Session 2: Z-Pay onboarding stored')
print()
print('⏱️ Waiting 30 seconds for memory pipeline to process...')

import time
time.sleep(30)
print('✅ Ready to query')


✅ Session 1: Acme Super onboarding stored
✅ Session 2: Z-Pay onboarding stored

⏱️ Waiting 30 seconds for memory pipeline to process...
✅ Ready to query


## Step 4: Query Each Memory Strategy

### Summary Strategy
Returns compressed session overviews. Best for: *"What did we discuss last time?"*

In [9]:
print('📋 SUMMARY STRATEGY')
print('   Query: "Acme Super trading platform and migration"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/summaries/acme-super-001',
    query='Acme Super trading platform and migration',
    top_k=2
)

if results:
    for r in results:
        print(f'\n  Score: {r["score"]:.2f}')
        print(f'  {r["content"]["text"][:500]}')
else:
    print('  (No summaries yet - may need more processing time)')


📋 SUMMARY STRATEGY
   Query: "Acme Super trading platform and migration"
------------------------------------------------------------

  Score: 0.60
          <topic name="Client Overview">
Acme Super is a superannuation fund managing $220 billion in assets, assigned as a new client to the user.
</topic>
        <topic name="Key Contacts">
        - CTO: John Chen (john.chen@acmesuper.com.au) — sponsoring the migration project.
- Backup contact: Sarah Liu, Head of Platform Engineering.
        </topic>
        <topic name="Current AWS Architecture">
        - Core trading platform runs on EC2 with Oracle on RDS.
- Streaming: Apache Kafka; Cachi


### User Preference Strategy
Extracts preferences and behavioral patterns. Best for: *"What does this client prefer?"*

In [11]:
print('💡 USER PREFERENCE STRATEGY')
print('   Query: "investment policy and risk appetite"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/preferences',
    query='investment policy and risk appetite',
    top_k=5
)

for r in results:
    print(f'\n  Score: {r["score"]:.2f}')
    print(f'  {r["content"]["text"][:300]}')


💡 USER PREFERENCE STRATEGY
   Query: "investment policy and risk appetite"
------------------------------------------------------------

  Score: 0.40
  {"context":"The user explicitly mentioned that Acme Super has an ESG-only investment policy, refuses exposure to fossil fuels, gambling, or weapons, reports to APRA quarterly, and has a moderate risk appetite.","preference":"Monitors and records client ESG policies, regulatory reporting obligations 

  Score: 0.35
  {"context":"Z-Pay has a disaster recovery setup in us-west-2 with a 5-minute RPO requirement.","preference":"Z-Pay的灾难恢复方案部署在us-west-2，RPO要求为5分钟","categories":["cloud computing","disaster recovery","infrastructure"]}

  Score: 0.35
  {"context":"The user explicitly mentioned that their client Acme Super requires sub-10ms latency for trade execution in ap-southeast-2, 99.99% availability, and a DR site in us-west-2 with 15-minute RPO. These are client requirements the user is tracking.","preference":"Tracks and prioritizes st

### Semantic Strategy
Extracts factual statements from **user messages only**. Best for: *"What are the hard facts?"*

⚠️ Only facts stated by the USER are stored — not assistant responses.

In [12]:
print('🧠 SEMANTIC STRATEGY')
print('   Query: "Acme Super AWS spend contacts architecture"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/facts/',
    query='Acme Super AWS spend contacts architecture',
    top_k=7
)

for r in results:
    print(f'\n  Score: {r["score"]:.2f}')
    print(f'  {r["content"]["text"]}')


🧠 SEMANTIC STRATEGY
   Query: "Acme Super AWS spend contacts architecture"
------------------------------------------------------------

  Score: 0.58
  Acme Super currently spends $1.2 million per month on AWS.

  Score: 0.57
  Acme Super's AWS spend breakdown: EC2 is 45% and RDS is 25%.

  Score: 0.52
  Acme Super's core trading platform runs on EC2 with Oracle on RDS.

  Score: 0.50
  Acme Super uses Kafka for streaming and Redis for caching.

  Score: 0.49
  Acme Super is a superannuation fund managing $220 billion in assets.

  Score: 0.48
  Acme Super's trading platform handles 50,000 transactions per second at peak.

  Score: 0.47
  Acme Super requires 99.99% availability.


## Building Memory-Enabled Agents

Now let's create a Strands Agent that automatically uses AgentCore Memory to provide personalized responses based on conversation history.

We'll build a Strands Agent with a Memory Hook to integrate with AgentCore Memory.


In [13]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def recall_client(query: str) -> str:
    '''Retrieve stored facts and preferences about a client.
    Args:
        query: What to recall (e.g., "Acme Super contacts" or "Z-Pay latency")
    '''
    all_results = []
    for ns in [f'fsi/{ACTOR_ID}/preferences', f'fsi/{ACTOR_ID}/facts/']:
        results = memory_client.retrieve_memories(
            memory_id=memory_id, namespace=ns, query=query, top_k=5
        )
        all_results.extend([r['content']['text'] for r in results if r['score'] > 0.3])
    return '\n'.join(all_results[:8]) if all_results else 'No context found.'

advisor = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are an FSI advisor. Always recall client context before answering. Cite specific facts.',
    tools=[recall_client],
)

print('✅ Memory-enabled agent ready')


✅ Memory-enabled agent ready


In [14]:
# Use Case 1: Specific client question
advisor('What is Acme Super\'s monthly AWS spend and what optimization opportunities exist?')


<thinking> To address the user's request, I need to retrieve specific facts about Acme Super's AWS spending and any known optimization opportunities. I will use the `recall_client` tool to gather this information. </thinking>

Tool #1: recall_client

Tool #2: recall_client
Acme Super currently spends $1.2 million per month on AWS. The breakdown shows that EC2 accounts for 45% of this spend and RDS accounts for 25%. The core trading platform runs on EC2 with Oracle on RDS.

An optimization opportunity identified is that Acme Super has zero Reserved Instances, which is considered a significant cost-saving opportunity. Implementing Reserved Instances could help reduce the overall AWS spend. Additionally, exploring the use of Graviton instances, as preferred by another client (Z-Pay) for cost savings, could also be beneficial.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Acme Super currently spends $1.2 million per month on AWS. The breakdown shows that EC2 accounts for 45% of this spend and RDS accounts for 25%. The core trading platform runs on EC2 with Oracle on RDS.\n\nAn optimization opportunity identified is that Acme Super has zero Reserved Instances, which is considered a significant cost-saving opportunity. Implementing Reserved Instances could help reduce the overall AWS spend. Additionally, exploring the use of Graviton instances, as preferred by another client (Z-Pay) for cost savings, could also be beneficial.'}], 'metadata': {'usage': {'inputTokens': 1502, 'outputTokens': 120, 'totalTokens': 1622}, 'metrics': {'latencyMs': 1295, 'timeToFirstByteMs': 470}}}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'recall_client': ToolMetrics(tool={'toolUseId': 'tooluse_2aZ7D6jAWJFTtdSkceMWZf', 'name': 'recall_client', 'input': {'query': 'Acme Super AWS op

In [15]:
# Use Case 2: Cross-client comparison
advisor('Compare the DR strategies and RPO targets for Acme Super vs Z-Pay.')


<thinking> To compare the disaster recovery (DR) strategies and Recovery Point Objective (RPO) targets for Acme Super and Z-Pay, I need to retrieve specific facts about their DR strategies and RPO targets. I will use the `recall_client` tool to gather this information. </thinking> 
Tool #3: recall_client

Tool #4: recall_client

Tool #5: recall_client

Tool #6: recall_client
Acme Super's DR strategy involves having a DR site in the us-west-2 region with a Recovery Point Objective (RPO) of 15 minutes. This means that in the event of a disaster, Acme Super aims to recover data with a maximum loss of 15 minutes.

Z-Pay's DR strategy is also hosted in the us-west-2 region, but with a stricter RPO of 5 minutes. This indicates that Z-Pay requires a more frequent data backup and recovery process to ensure minimal data loss in case of a disaster.

In summary, while both clients have their DR sites in the same region, Z-Pay has a more stringent RPO requirement compared to Acme Super.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Acme Super's DR strategy involves having a DR site in the us-west-2 region with a Recovery Point Objective (RPO) of 15 minutes. This means that in the event of a disaster, Acme Super aims to recover data with a maximum loss of 15 minutes.\n\nZ-Pay's DR strategy is also hosted in the us-west-2 region, but with a stricter RPO of 5 minutes. This indicates that Z-Pay requires a more frequent data backup and recovery process to ensure minimal data loss in case of a disaster.\n\nIn summary, while both clients have their DR sites in the same region, Z-Pay has a more stringent RPO requirement compared to Acme Super."}], 'metadata': {'usage': {'inputTokens': 3646, 'outputTokens': 147, 'totalTokens': 3793}, 'metrics': {'latencyMs': 1690, 'timeToFirstByteMs': 434}}}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'recall_client': ToolMetrics(tool={'toolUseId': 'tooluse_UMNIB6PwaS2jBseYRYNVZQ', 'name': '

---

## Step 6: Session Handover

The most powerful FSI use case: a **new advisor takes over** the account with zero prior context. The agent provides a full briefing from memory.

In [16]:
# Simulate: completely new session, new advisor, no conversation history
new_advisor = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are helping a new advisor prepare for their first client meeting. Provide a comprehensive briefing using stored context.',
    tools=[recall_client],
)

new_advisor('I just took over the Acme Super account. Brief me on everything: contacts, architecture, requirements, spend, migration plans, and any risks.')


<thinking> To prepare for the first client meeting with the Acme Super account, I need to gather comprehensive information about the client. This includes contacts, architecture, requirements, spend, migration plans, and any risks associated with the account. I will use the `recall_client` tool to retrieve this information. </thinking>

Tool #1: recall_client

Tool #2: recall_client

Tool #3: recall_client

Tool #4: recall_client

Tool #5: recall_client

Tool #6: recall_client
Here's a comprehensive briefing for your first client meeting with Acme Super:

### Contacts
- **Primary Contact**: Not specified in the stored context. You may need to reach out to the previous advisor or the client directly to obtain this information.

### Architecture
- **Transaction Volumes**: 50,000 transactions per second (TPS).
- **Streaming**: Uses Kafka for streaming data.
- **Caching**: Utilizes Redis for caching.
- **Migration Targets**: Plans to migrate to EKS (Elastic Kubernetes Service) with Aurora 

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's a comprehensive briefing for your first client meeting with Acme Super:\n\n### Contacts\n- **Primary Contact**: Not specified in the stored context. You may need to reach out to the previous advisor or the client directly to obtain this information.\n\n### Architecture\n- **Transaction Volumes**: 50,000 transactions per second (TPS).\n- **Streaming**: Uses Kafka for streaming data.\n- **Caching**: Utilizes Redis for caching.\n- **Migration Targets**: Plans to migrate to EKS (Elastic Kubernetes Service) with Aurora PostgreSQL by Q3 2026.\n\n### Requirements\n- **Latency**: Requires sub-10ms latency for trade execution in the `ap-southeast-2` region.\n- **Availability**: Demands 99.99% availability.\n- **Disaster Recovery (DR)**: Needs a DR site in `us-west-2` with a 15-minute Recovery Point Objective (RPO).\n\n### Spend\n- **Monthly AWS Spend**: $1.2 million.\n- **Reserved Instances**: Current

In [17]:
# New advisor asks about the other client too
new_advisor('What about Z-Pay? What are their key concerns and who do I contact?')


<thinking> To provide a comprehensive briefing on Z-Pay, I need to gather information about their key concerns, contacts, architecture, requirements, spend, migration plans, and any risks. I will use the `recall_client` tool to retrieve this information. </thinking> 
Tool #7: recall_client

Tool #8: recall_client

Tool #9: recall_client

Tool #10: recall_client

Tool #11: recall_client

Tool #12: recall_client
Here's a comprehensive briefing on Z-Pay:

### Contacts
- **Main Contact**: David Park, VP Engineering, reachable at david.park@zpay.com.

### Key Concerns
- **Data Sovereignty**: Z-Pay's legal team requires all transaction data to remain in Australia due to upcoming ASIC regulations on Buy Now, Pay Later (BNPL). This means all data must stay within the `ap-southeast-2` region.
- **Fraud Detection**: Z-Pay processes 5 million transactions daily and requires real-time fraud detection with latency under 100ms using AWS SageMaker with custom models.
- **Disaster Recovery**: Z-Pay ha

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's a comprehensive briefing on Z-Pay:\n\n### Contacts\n- **Main Contact**: David Park, VP Engineering, reachable at david.park@zpay.com.\n\n### Key Concerns\n- **Data Sovereignty**: Z-Pay's legal team requires all transaction data to remain in Australia due to upcoming ASIC regulations on Buy Now, Pay Later (BNPL). This means all data must stay within the `ap-southeast-2` region.\n- **Fraud Detection**: Z-Pay processes 5 million transactions daily and requires real-time fraud detection with latency under 100ms using AWS SageMaker with custom models.\n- **Disaster Recovery**: Z-Pay has a disaster recovery setup in `us-west-2` with a 5-minute Recovery Point Objective (RPO) requirement.\n\n### Architecture\n- **Transaction Volumes**: Processes 5 million transactions daily.\n- **Cost Optimization**: Prefers Graviton instances for cost savings.\n- **AWS Spend**: Spends $800,000 per month on AWS.\n\n#

## Examining the Agent Loop

The agent loop is how Strands Agents process requests:

1. **Receive** user input
2. **Reason** using the LLM (decide what to do)
3. **Act** by calling a tool
4. **Observe** the tool result
5. **Repeat** or respond to the user

The table below shows each message in the loop — what the model said, which tool it called, and what result it received:

In [18]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f'Agent Loop Cycles: {advisor.event_loop_metrics.cycle_count}' if hasattr(agent, 'event_loop_metrics') else '')

# Get the last agent used in this notebook
active_agent = [v for v in dir() if not v.startswith('_')]

table = Table(title='Agent Messages', show_lines=True)
table.add_column('Role', style='green', width=10)
table.add_column('Text', style='magenta', max_width=50)
table.add_column('Tool', style='cyan', width=20)
table.add_column('Input', style='cyan', max_width=30)
table.add_column('Result', style='cyan', max_width=30)

for msg in advisor.messages[-6:]:  # Show last 6 messages for readability
    text = [c['text'] for c in msg['content'] if 'text' in c]
    tool_name = [c['toolUse']['name'] for c in msg['content'] if 'toolUse' in c]
    tool_input = [c['toolUse']['input'] for c in msg['content'] if 'toolUse' in c]
    tool_result = [c['toolResult']['content'][0] for c in msg['content'] if 'toolResult' in c]
    table.add_row(
        msg['role'],
        (text[-1][:100] + '...') if text and len(text[-1]) > 100 else (text[-1] if text else ''),
        tool_name[-1] if tool_name else '',
        (json.dumps(tool_input[-1])[:80] + '...') if tool_input else '',
        (json.dumps(tool_result[-1])[:80] + '...') if tool_result else '',
    )

console.print(table)


Agent Loop Cycles: 4

                                                  Agent Messages                                                   
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role       ┃ Text                    ┃ Tool                 ┃ Input                   ┃ Result                  ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user       │                         │                      │                         │ {"text":                │
│            │                         │                      │                         │ "{\"context\":\"The     │
│            │                         │                      │                         │ user explicitly         │
│            │                         │                      │                         │ identified that Acme    │
│            │                         │                      │                         │ Super spends $...       │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ Acme Super currently    │                      │                         │                         │
│            │ spends $1.2 million per │                      │                         │                         │
│            │ month on AWS. The       │                      │                         │                         │
│            │ breakdown shows that    │                      │                         │                         │
│            │ EC2 accounts for...     │                      │                         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ user       │ Compare the DR          │                      │                         │                         │
│            │ strategies and RPO      │                      │                         │                         │
│            │ targets for Acme Super  │                      │                         │                         │
│            │ vs Z-Pay.               │                      │                         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ <thinking> To compare   │ recall_client        │ {"query": "Z-Pay        │                         │
│            │ the disaster recovery   │                      │ RPO"}...                │                         │
│            │ (DR) strategies and     │                      │                         │                         │
│            │ Recovery Point          │                      │                         │                         │
│            │ Objective (RPO)         │                      │                         │                         │
│            │ targe...                │                      │                         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ user       │                         │                      │                         │ {"text":                │
│            │                         │                      │                         │ "{\"context\":\"Z-Pay   │
│            │                         │                      │                         │ has a disaster recovery │
│            │                         │                      │                         │ setup in us-west-2 with │
│            │                         │                      │                         │ a...                    │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ Acme Super's DR         │                

---

## Cleanup (Optional)

In [19]:
# Delete memory when done
memory_client.delete_memory_and_wait(memory_id=memory_id)
print('✅ Memory deleted')


✅ Memory deleted


## Summary

| What We Did | FSI Value |
|------------|----------|
| Stored client conversations | Build institutional knowledge |
| Summary strategy | Quick session recaps for follow-ups |
| Preference strategy | Personalized recommendations (ESG, risk) |
| Semantic strategy | Hard facts recall (spend, contacts, dates) |
| Memory-enabled agent | Context-aware responses without re-asking |
| Session handover | New advisor gets full briefing instantly |

### Key Insight

Memory turns a stateless AI tool into a **relationship-aware advisor** that accumulates knowledge over time — exactly what FSI clients expect from their support team.